# Exercise 4 — Testing an API

**After:** [Day 5](../lessons/day5-testing-apis/09_theory_testing_apis.md)

These run inline with `TestClient` so you can iterate fast. The project's real suite lives in
`project-insight-api/tests/`.

In [ ]:
import pytest
from fastapi import Depends, FastAPI, HTTPException, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel

# --- the app under test -------------------------------------------------
def get_store():
    raise RuntimeError("no store configured")      # production would build a real one

app = FastAPI()

class ItemOut(BaseModel):
    id: int
    name: str

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/items", response_model=list[ItemOut])
def list_items(limit: int = Query(default=10, ge=1, le=100), store = Depends(get_store)):
    return store.all()[:limit]

@app.get("/items/{item_id}", response_model=ItemOut)
def read_item(item_id: int, store = Depends(get_store)):
    found = [i for i in store.all() if i["id"] == item_id]
    if not found:
        raise HTTPException(404, detail=f"No item {item_id}")
    return found[0]

print("app ready")

## ⭐ Level 1 — Swap the store

Write a fake store returning three items (each with `id`, `name`, **and a `secret` field**), install
it with `dependency_overrides`, and prove:

1. `/items` returns 3 items
2. `/items/1` returns the right one
3. `/items/999` returns 404
4. `secret` never appears in any response

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
ITEMS = [
    {"id": 1, "name": "alpha", "secret": "do-not-publish"},
    {"id": 2, "name": "beta",  "secret": "do-not-publish"},
    {"id": 3, "name": "gamma", "secret": "do-not-publish"},
]

class FakeStore:
    def all(self):
        return ITEMS

def fake_store():
    yield FakeStore()

app.dependency_overrides[get_store] = fake_store
client = TestClient(app)

assert len(client.get("/items").json()) == 3
assert client.get("/items/1").json()["name"] == "alpha"
assert client.get("/items/999").status_code == 404

for row in client.get("/items").json():
    assert "secret" not in row
assert set(client.get("/items/1").json()) == {"id", "name"}
print("all four hold")

app.dependency_overrides.clear()
```

The `secret` field never leaves, and no handler filters it — `response_model=ItemOut` does, because
it declares only `id` and `name`. That's why the fourth assertion uses an **exact key set**: here the
complete shape *is* the promise.
</details>

## ⭐⭐ Level 2 — Parametrize the boundaries

Write one `parametrize`d test covering the `limit` bounds: `0`, `-1`, `101`, `"abc"` must all be
**422**, while `1`, `10` and `100` must be **200**. Run it with real pytest semantics by asserting in
a loop, and make the failure message name the offending value.

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
app.dependency_overrides[get_store] = fake_store
client = TestClient(app)

cases = [(0, 422), (-1, 422), (101, 422), ("abc", 422), (1, 200), (10, 200), (100, 200)]

for limit, expected in cases:
    r = client.get("/items", params={"limit": limit})
    assert r.status_code == expected, (
        f"limit={limit!r}: expected {expected}, got {r.status_code} - {r.text[:80]}"
    )
print(f"all {len(cases)} boundary cases behave")

app.dependency_overrides.clear()
```

As a real pytest file it becomes:

```python
@pytest.mark.parametrize(("limit", "expected"), cases)
def test_limit_bounds(client, limit, expected):
    assert client.get("/items", params={"limit": limit}).status_code == expected
```

`parametrize` reports each case as a **separate test**, so a failure names the exact input instead of
"the limit test failed". That's the whole reason to prefer it over a loop in a real suite.

Notice `100` and `101`: testing the value *at* the boundary and the value *just past* it is where
off-by-one bugs actually live. Testing `50` proves very little.
</details>

## ⭐⭐⭐ Level 3 — Catch a bug your tests currently miss

The `/items` endpoint has a bug: `limit` is applied **in Python**, after the store has returned
everything. With a million rows that's a memory problem, and it means `limit` never reaches the query.

1. Write a test that **fails** on the current code by asserting the store was asked to limit.
2. Fix the endpoint so the store receives the limit.
3. Show your test now passes.

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
class RecordingStore:
    def __init__(self):
        self.calls = []
    def all(self, limit=None):
        self.calls.append({"limit": limit})
        return ITEMS[:limit] if limit else ITEMS

recorder = RecordingStore()

def recording_store():
    yield recorder

# --- 1. the failing test -------------------------------------------------
app.dependency_overrides[get_store] = recording_store
client = TestClient(app)

recorder.calls.clear()
client.get("/items", params={"limit": 2})
try:
    assert recorder.calls[0]["limit"] == 2, f"store was asked for {recorder.calls[0]}"
    print("1. unexpectedly passed")
except AssertionError as exc:
    print("1. FAILS as expected:", exc)

# --- 2. the fix ----------------------------------------------------------
# NOTE: re-decorating "/items" on the SAME app does NOT replace the old route -
# FastAPI appends it, and first match wins (Day 2). Build a fresh app instead.
fixed_app = FastAPI()

@fixed_app.get("/items", response_model=list[ItemOut])
def list_items_fixed(limit: int = Query(default=10, ge=1, le=100), store = Depends(get_store)):
    return store.all(limit=limit)             # push the limit DOWN to the store

fixed_app.dependency_overrides[get_store] = recording_store
client = TestClient(fixed_app)
recorder.calls.clear()
r = client.get("/items", params={"limit": 2})

# --- 3. now it passes ----------------------------------------------------
assert r.status_code == 200
assert len(r.json()) == 2
assert recorder.calls[0]["limit"] == 2
print("3. passes:", recorder.calls)

app.dependency_overrides.clear()
fixed_app.dependency_overrides.clear()
```

> ⚠️ **The trap inside the fix.** Re-running `@app.get("/items")` on the same app object does not
> overwrite the first route — it appends a second one, and FastAPI matches in definition order, so
> the *original* buggy handler keeps serving. That is the Day 2 route-order rule showing up somewhere
> you would not expect it. When you want to replace a route in an experiment, build a fresh app.

A **recording fake** lets you assert on the *interaction*, not just the result — which is the only way
to catch this class of bug, because the response body looks identical either way.

Use the technique sparingly. Asserting on exact SQL strings couples tests to your implementation, so a
harmless rewrite turns the suite red. Assert on **what was asked for** (a limit of 2 reached the
store), never on **how it was phrased**.

This is exactly the bug the project's `insight-api` avoids by putting `LIMIT :limit` in the SQL rather
than slicing a list in Python.
</details>